In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length, when, count, avg, round as spark_round

# spark i baslatiyorum
spark = SparkSession.builder \
    .appName('TwitterAnalysis') \
    .getOrCreate()

print("Spark basladi!")

dosya = '../data/processed/cleaned_tweets.csv'

# csv yi pyspark dataframe olarak oku
# quote, escape ve multiLine parametreleri tweet metinlerindeki virgul ve satirbasi sorunlarini onluyor
spark_df = spark.read.csv(
    dosya,
    header=True,
    inferSchema=True,
    quote='"',
    escape='"',
    multiLine=True
)

# semasina bakalim hangi sutun ne tipte
spark_df.printSchema()

In [ ]:
# filter(): satirlari kosullara gore filtreler (SQL WHERE gibi)
# isin(): belirtilen degerlerden herhangi birine esit mi?
# col(): sutun adina referans verir
# &: VE (AND) operatoru — tum kosullar saglanmali
# select(): sadece istedigim sutunlari sec (SQL SELECT gibi)
print("Islem 1: United/American negatif + guven>0.9:")
filtreli = spark_df.filter(
    (col("airline").isin("United", "American")) &
    (col("airline_sentiment") == "negative") &
    (col("airline_sentiment_confidence") > 0.9)
).select("airline", "negativereason", "airline_sentiment_confidence", "text")
filtreli.show(3, truncate=50)

In [ ]:
# withColumn(): mevcut DataFrame'e yeni sutun ekler
# when(): kosul saglanirsa bu degeri ver
# otherwise(): hicbir kosul saglanmazsa bu degeri ver
# zincirleme when kullanabiliyorum — ilk saglanan kosul gecerli olur
print("Islem 2: Guven kategorisi sutunu:")
df_kategorili = spark_df.withColumn(
    "guven_kategori",
    when(col("airline_sentiment_confidence") == 1.0, "kesin")
    .when(col("airline_sentiment_confidence") >= 0.8, "yuksek")
    .otherwise("dusuk")
)
df_kategorili.select("airline", "airline_sentiment_confidence", "guven_kategori").show(5)

In [ ]:
# groupBy(): belirtilen sutuna gore grupla
# agg(): birden fazla aggregation fonksiyonunu ayni anda calistir
# alias(): sonuc sutununa isim ver
# orderBy(): sirala — desc() ile azalan
print("Islem 3: Havayolu bazli istatistikler:")
coklu_agg = spark_df.groupBy("airline").agg(
    count("*").alias("tweet_sayisi"),
    spark_round(avg("airline_sentiment_confidence"), 3).alias("ort_guven"),
    avg("retweet_count").alias("ort_rt")
).orderBy(col("tweet_sayisi").desc())
coklu_agg.show()

In [ ]:
# UDF: User Defined Function — kendi yazdigim Python fonksiyonunu Spark'ta kullanmami saglar
# udf() ile fonksiyonu Spark'in anlayacagi formata ceviriyorum
# StringType(): fonksiyonun dondurdugu deger tipi (string)
# Neden UDF? Bazi islemler Spark'in hazir fonksiyonlariyla yapilamaz,
# bu durumda kendi fonksiyonumuzu yazariz

def tweet_uzunluk_kategorisi(text):
    """tweet uzunluguna gore kisa/orta/uzun etiketi ver"""
    if text is None:
        return "bilinmiyor"
    uzunluk = len(text)
    if uzunluk < 60:
        return "kisa"
    elif uzunluk < 120:
        return "orta"
    else:
        return "uzun"

# fonksiyonu UDF olarak kaydet
uzunluk_udf = udf(tweet_uzunluk_kategorisi, StringType())

# yeni sutun olustur
print("Islem 4: Tweet uzunluk kategorisi (UDF):")
df_uzunluk = spark_df.withColumn("uzunluk_kat", uzunluk_udf(col("text")))
df_uzunluk.groupBy("uzunluk_kat").count().show()

In [ ]:
# Window: satirlari gruplar icinde siralamak veya hesaplama yapmak icin
# partitionBy: hangi alana gore grupla (her havayolu kendi icinde)
# orderBy: gruplarin icinde neye gore sirala
# row_number(): her satira sirasina gore numara ver (1, 2, 3, ...)
# boylece "her havayolunun en yuksek guvenli tweetleri" gibi sorulari cevaplayabilirim

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

print("Islem 5: Window function ile havayolu icinde siralama:")
pencere = Window.partitionBy("airline").orderBy(col("airline_sentiment_confidence").desc())

df_sirali = spark_df.withColumn("sira", row_number().over(pencere))

# her havayolunun en yuksek guvenli ilk 2 tweetini goster
df_sirali.filter(col("sira") <= 2) \
    .select("airline", "sira", "airline_sentiment_confidence", "airline_sentiment") \
    .orderBy("airline", "sira") \
    .show(12)

In [ ]:
# count(): toplam satir sayisini verir — bu bir ACTION, yani hemen calisir
# distinct(): benzersiz degerleri bul
# distinct().count(): benzersiz degerlerin sayisini ver
toplam = spark_df.count()
benzersiz_kullanici = spark_df.select("name").distinct().count()
benzersiz_konum = spark_df.filter(col("tweet_location") != "Bilinmiyor") \
    .select("tweet_location").distinct().count()
print(f"Toplam kayit: {toplam}")
print(f"Benzersiz kullanici: {benzersiz_kullanici}")
print(f"Benzersiz konum (bilinmiyor haric): {benzersiz_konum}")

In [ ]:
# describe(): count, mean, stddev, min, max degerlerini hesaplar
# birden fazla sutun icin ayni anda istatistik cikariyorum
print("Istatistiksel ozet:")
spark_df.describe(['airline_sentiment_confidence', 'retweet_count']).show()

In [ ]:
# createOrReplaceTempView: Spark DataFrame'i SQL tablosu gibi kullanmami saglar
# CASE WHEN: kosullu degerlendirme — "eger sentiment negative ise 1 yaz, degilse 0 yaz"
# SUM(CASE WHEN...): kosulu saglayanlarin sayisini hesaplar
# HAVING: GROUP BY sonrasinda filtre uygular (WHERE gruplamadan ONCE, HAVING gruplamadan SONRA)
spark_df.createOrReplaceTempView('tweets_table')

print("SQL 1: Duygu dagilimi tablosu:")
sql1 = spark.sql("""
    SELECT
        airline,
        SUM(CASE WHEN airline_sentiment = 'negative' THEN 1 ELSE 0 END) as negatif,
        SUM(CASE WHEN airline_sentiment = 'positive' THEN 1 ELSE 0 END) as pozitif,
        SUM(CASE WHEN airline_sentiment = 'neutral' THEN 1 ELSE 0 END) as notr,
        COUNT(*) as toplam
    FROM tweets_table
    GROUP BY airline
    HAVING COUNT(*) > 500
    ORDER BY negatif DESC
""")
sql1.show()

In [ ]:
# Subquery: ic ice sorgu — bir sorgunun sonucunu baska bir sorguda kullan
# ic sorgu (alt_sorgu): her havayolunun negatif tweet yuzdesini hesaplar
# dis sorgu: ic sorgunun sonucunu filtreleyerek genel ortalamadan kotu olanlari buluyor
# WHERE ... > (SELECT ...): dis filtre icindeki subquery genel ortalamayi hesaplar
print("SQL 2: Ortalamanin ustunde negatif oranina sahip havayollari:")
sql2 = spark.sql("""
    SELECT
        airline,
        ROUND(negatif_oran, 2) as negatif_yuzde
    FROM (
        SELECT
            airline,
            (SUM(CASE WHEN airline_sentiment = 'negative' THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) as negatif_oran
        FROM tweets_table
        GROUP BY airline
    ) alt_sorgu
    WHERE negatif_oran > (
        SELECT SUM(CASE WHEN airline_sentiment = 'negative' THEN 1 ELSE 0 END) * 100.0 / COUNT(*)
        FROM tweets_table
    )
    ORDER BY negatif_yuzde DESC
""")
sql2.show()

In [ ]:
import pandas as pd
import time

# ayni veriyi pandas ile de okuyalim
dosya = '../data/processed/cleaned_tweets.csv'
pandas_df = pd.read_csv(dosya)

# --- Pandas testi ---
# negatif tweetleri filtrele + havayoluna gore grupla + sayi ve ortalama hesapla
t1 = time.time()
pandas_sonuc = pandas_df[pandas_df['airline_sentiment'] == 'negative'].groupby('airline').agg(
    adet=('airline_sentiment', 'count'),
    ort_guven=('airline_sentiment_confidence', 'mean')
)
t2 = time.time()
pandas_sure = t2 - t1

# --- PySpark testi ---
# ayni islem: filtrele + grupla + say + ortalama al + collect ile sonuc topla
t3 = time.time()
spark_sonuc = spark_df.filter(col("airline_sentiment") == "negative") \
    .groupBy("airline") \
    .agg(
        count("*").alias("adet"),
        avg("airline_sentiment_confidence").alias("ort_guven")
    ).collect()
t4 = time.time()
spark_sure = t4 - t3

print(f"Pandas suresi:  {pandas_sure:.5f} sn")
print(f"PySpark suresi: {spark_sure:.5f} sn")

In [ ]:
spark.stop()